# Formatear Salida de Requerimientos

Lee el CSV de salida del `Assigner_Requirements.py`, lo reconstruye correctamente
y lo exporta como `.xlsx` para visualización limpia en Excel.

**Problema:** El CSV usa `;` como separador, pero los campos de descripción también
contienen `;`, lo que desalinea las columnas al abrir directamente en Excel.

**Solución:** Reconstruir el DataFrame uniendo el input original con las columnas
predichas del output, y exportar a `.xlsx`.

## 1. Librerías

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])
print('openpyxl listo.')

In [ ]:
import os
import glob
import pandas as pd
from pathlib import Path
from datetime import datetime

RAIZ      = Path(os.getcwd()).parent
DIR_SALIDA = RAIZ / 'Salida'
DIR_ENTRADA = RAIZ / 'Entrada'
print('Raíz:', RAIZ)

## 2. Seleccionar el CSV de salida más reciente

In [ ]:
archivos = sorted(glob.glob(str(DIR_SALIDA / 'requerimientos_con_asignacion_*.csv')))

if not archivos:
    raise FileNotFoundError(f'No se encontró ningún CSV en {DIR_SALIDA}. '
                             'Ejecuta primero Assigner_Requirements.py.')

ruta_salida = archivos[-1]  # el más reciente
print(f'Archivo seleccionado: {ruta_salida}')

## 3. Reconstruir el DataFrame

El CSV de salida puede tener filas malformadas porque los campos de descripción
contienen `;`. La estrategia es:

1. Leer el **input original** (`sc_req_item.csv`) que tiene formato limpio.
2. Leer el output solo para extraer las **columnas predichas** (`Clasificación`,
   `assigned_to`, `predicted_assigned_to`, `fecha_resolucion`).
3. Unir ambos por `number`.

In [ ]:
# ── Leer input original ──────────────────────────────────────────────────────
ruta_input = DIR_ENTRADA / 'sc_req_item.csv'

df_input = None
for enc in ('utf-8-sig', 'latin-1'):
    for sep in (',', ';'):
        try:
            df_input = pd.read_csv(ruta_input, encoding=enc, sep=sep, dtype=str)
            if len(df_input.columns) > 2:
                print(f'Input leído — encoding={enc}, sep={sep!r}, filas={len(df_input)}')
                break
        except Exception:
            continue
    if df_input is not None and len(df_input.columns) > 2:
        break

print(f'Columnas input: {list(df_input.columns)}')

In [ ]:
# ── Leer output: solo las primeras N columnas fijas (antes de description) ───
# Las columnas predichas están al final del CSV; las leemos extrayendo
# las últimas columnas por posición usando el número total de columnas del input
# más las columnas añadidas.

COLS_PREDICHAS = ['Clasificación', 'predicted_assigned_to', 'predicted_assignment_group',
                  'assigned_to', 'assignment_group', 'fecha_resolucion']

try:
    # Intentar lectura normal — puede fallar en filas con ; en description
    df_out = pd.read_csv(ruta_salida, sep=';', encoding='latin-1',
                         dtype=str, engine='python', on_bad_lines='skip')
    cols_predichas_presentes = [c for c in COLS_PREDICHAS if c in df_out.columns]
    print(f'Output leído — filas válidas: {len(df_out)}')
    print(f'Columnas predichas encontradas: {cols_predichas_presentes}')
except Exception as e:
    print(f'Error leyendo output: {e}')
    df_out = pd.DataFrame()

In [ ]:
# ── Unir input original con columnas predichas ───────────────────────────────
if not df_out.empty and 'number' in df_out.columns and cols_predichas_presentes:
    df_pred = df_out[['number'] + cols_predichas_presentes].drop_duplicates('number')
    df_final = df_input.merge(df_pred, on='number', how='left', suffixes=('_orig', ''))
    # Si assigned_to ya existía en el input, la del output (predicha) tiene prioridad
    if 'assigned_to_orig' in df_final.columns:
        df_final = df_final.drop(columns=['assigned_to_orig'])
    if 'assignment_group_orig' in df_final.columns:
        df_final = df_final.drop(columns=['assignment_group_orig'])
    print(f'DataFrame final: {len(df_final)} filas × {len(df_final.columns)} columnas')
else:
    print('No se pudo unir — usando output directo.')
    df_final = df_out

df_final.head(3)

## 4. Seleccionar y ordenar columnas de interés

In [ ]:
# Orden preferido de columnas en el Excel final
ORDEN_COLS = [
    'number',
    'state',
    'opened_at',
    'ref_sc_req_item.requested_for',
    'requested_for.title',
    'requested_for.company',
    'short_description',
    'description',
    'Clasificación',
    'predicted_assigned_to',
    'fecha_resolucion',
]

# Solo incluir las que existen
cols_export = [c for c in ORDEN_COLS if c in df_final.columns]

# Agregar columnas extra que no estén en el orden definido (por si acaso)
extras = [c for c in df_final.columns if c not in cols_export
          and c not in ('predicted_assignment_group', 'short_norm', 'desc_norm',
                        'short_core', 'desc_core', 'texto_limpio')]
cols_export += extras

df_export = df_final[cols_export].copy()
print(f'Columnas a exportar ({len(cols_export)}): {cols_export}')

## 5. Exportar a Excel

In [ ]:
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
ruta_xlsx = DIR_SALIDA / f'requerimientos_asignados_{timestamp}.xlsx'

with pd.ExcelWriter(ruta_xlsx, engine='openpyxl') as writer:
    df_export.to_excel(writer, index=False, sheet_name='Asignaciones')

    # Autoajustar ancho de columnas
    ws = writer.sheets['Asignaciones']
    for col_cells in ws.columns:
        max_len = max(
            (len(str(cell.value)) if cell.value is not None else 0)
            for cell in col_cells
        )
        col_letter = col_cells[0].column_letter
        ws.column_dimensions[col_letter].width = min(max_len + 4, 60)

print(f'Excel exportado: {ruta_xlsx}')
print(f'Total filas: {len(df_export)}')
df_export.head(10)

## 6. Resumen de asignaciones

In [ ]:
if 'Clasificación' in df_export.columns:
    print('── Distribución por Clasificación ──')
    print(df_export['Clasificación'].value_counts().to_string())

print()

if 'predicted_assigned_to' in df_export.columns:
    print('── Distribución por Asignado ──')
    print(df_export['predicted_assigned_to'].value_counts().to_string())